In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms, models
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

XRAY_DIR = r"C:\Users\asus\Documents\project\medreport-analyzer\data\chest_xray"
CT_DIR = r"C:\Users\asus\Documents\project\medreport-analyzer\data\ct_scan"

BATCH_SIZE = 32
NUM_EPOCHS = 8
LEARNING_RATE = 0.0001

Using device: cuda


In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load each modality's images, ignoring their disease labels --
# we only care about "this came from the xray folder" vs "this came from the ct folder"
xray_train = datasets.ImageFolder(f"{XRAY_DIR}/train", transform=transform)
xray_test = datasets.ImageFolder(f"{XRAY_DIR}/test", transform=transform)
xray_val = datasets.ImageFolder(f"{XRAY_DIR}/val", transform=transform)
ct_all = datasets.ImageFolder(CT_DIR, transform=transform)

class ModalityDataset(torch.utils.data.Dataset):
    """Wraps an existing ImageFolder dataset but replaces its label with a fixed modality id."""
    def __init__(self, base_dataset, modality_label):
        self.base_dataset = base_dataset
        self.modality_label = modality_label

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        return image, self.modality_label

MODALITY_LABELS = {"xray": 0, "ct_scan": 1}  # add "mri_scan": 2 later

xray_combined = ConcatDataset([xray_train, xray_test, xray_val])
full_dataset = ConcatDataset([
    ModalityDataset(xray_combined, MODALITY_LABELS["xray"]),
    ModalityDataset(ct_all, MODALITY_LABELS["ct_scan"]),
])

print("Total images:", len(full_dataset))

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Total images: 8337
Train: 5835 | Val: 1250 | Test: 1252


In [3]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(MODALITY_LABELS))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

best_val_acc = 0.0
for epoch in range(NUM_EPOCHS):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    elapsed = time.time() - start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Time: {elapsed:.1f}s")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "modality_classifier_best.pt")
        print("  -> Saved new best model")

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Final Test Accuracy: {test_acc:.4f}")